In [ ]:
import torch
import wandb
import json

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset

# Model name
model_name = "distilbert-base-cased"

# Tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

# Model
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("Tokenizer and Model Loaded Successfully")

# WandB login
wandb.init(
    project="mlops-assignment2",
    name="distilbert-task5"
)

print("WandB Initialized Successfully")

# Metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

print("Metrics Function Created Successfully")

# Dummy dataset
dummy_data = {
    "input_ids": [[101, 2023, 2003, 1037, 2742, 102]],
    "attention_mask": [[1, 1, 1, 1, 1, 1]],
    "labels": [1]
}

train_dataset = Dataset.from_dict(dummy_data)

print("Dummy Dataset Created Successfully")

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    warmup_steps=100,

    weight_decay=0.01,

    logging_steps=50,

    save_strategy="no",

    load_best_model_at_end=False,

    report_to="wandb",

    run_name="distilbert-run-1"
)

print("Training Arguments Created Successfully")

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,

    compute_metrics=compute_metrics
)

print("Trainer Created Successfully")

In [ ]:
import json
import wandb

from sklearn.metrics import classification_report

# Run evaluation
eval_results = trainer.evaluate(eval_dataset=train_dataset)

print(eval_results)

# Log metrics to W&B
wandb.log({
    "final/loss": eval_results["eval_loss"],
    "final/accuracy": eval_results["eval_accuracy"],
    "final/f1": eval_results["eval_f1"],
})

# Predictions
preds = trainer.predict(train_dataset).predictions.argmax(-1)

# Labels
labels = [item["labels"] for item in train_dataset]

# Classification report
report = classification_report(
    labels,
    preds,
    output_dict=True
)

# Save report
with open("eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

# Create W&B artifact
artifact = wandb.Artifact(
    "eval-report",
    type="evaluation"
)

artifact.add_file("eval_report.json")

wandb.log_artifact(artifact)

wandb.finish()

print("Task 5 Completed Successfully")